In [ ]:
import sys
sys.path.append('../')
import time

import h5py
import numpy as np
from matplotlib import pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

Script for perform the CARCARA X-ray beam scan of its caustic

Initialize cax control of siriuspy devices

In [17]:
cax = CAXCtrl()

# Functions

Useful functions for the scan

In [ ]:
def current_position():
    return cax.dvf_B1.z_pos


def move_screen(z_pos):
    cax.dvf_B1.z_pos = z_pos

def move_robust_screen(z_pos):
    move_screen(z_pos)
    while(np.abs(cax.dvf_B1.z_pos - z_pos) >= 0.2):
        time.sleep(0.5)
    time.sleep(0.5)

# Scan

Initial state

In [ ]:
position0 = current_position()

with open('initial_screen_pos.txt','w') as f:
    f.write(position0)

print(position0)

Parameters

In [15]:
pos_start = 560
pos_end   = 140
pos_step  = -20

positions = np.arange(pos_start,pos_end+pos_step,pos_step)

Initialize file

In [ ]:
filename = 'caustic_scan_20250717_01.h5'
filedir = "/home/ids/data/2025-07-17-Caustic"
file = HDF5File(filename=filename,filedir=filedir)

file.save_metadata(metadata_dict={
    'position': cax.dvf_B1.z_pos
})

Execution

In [ ]:

t0 = time.time()

for i, pos in enumerate(positions):

    move_robust_screen(z_pos=pos)

    screen_pos = cax.dvf_B1.z_pos
    print('finished movement {0}/{1}: pos={2:.1f} mm'.format(i+1, len(positions), pos), end='\r')

    img = cax.dvf_B1.image

    pos_dict = {'position': cax.dvf_B1.z_pos}

    file.save_dataset(dsetname=f'scan-{i:03d}',dsetdata=img,dsetmetadata=pos_dict)

t1 = time.time()

print(f'elapsed time [s]: {t1-t0}')

finished scan!ent 22/22: pos=140.0 mm
